![logo](https://raw.githubusercontent.com/sciknoworg/OntoAligner/main/images/logo-with-background.png)

[![PyPI version](https://badge.fury.io/py/OntoAligner.svg)](https://badge.fury.io/py/OntoAligner)
[![PyPI Downloads](https://static.pepy.tech/badge/ontoaligner)](https://pepy.tech/projects/ontoaligner)
![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)
[![pre-commit](https://img.shields.io/badge/pre--commit-enabled-brightgreen?logo=pre-commit)](https://github.com/pre-commit/pre-commit)
[![Documentation Status](https://readthedocs.org/projects/ontoaligner/badge/?version=main)](https://ontoaligner.readthedocs.io/)
[![Maintenance](https://img.shields.io/badge/Maintained%3F-yes-green.svg)](MAINTANANCE.md)
 [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.14533133.svg)](https://doi.org/10.5281/zenodo.14533133)

- **Documentation website**: [https://ontoaligner.readthedocs.io/index.html](https://ontoaligner.readthedocs.io/index.html)
- **Resource Paper**: [https://doi.org/10.1007/978-3-031-94578-6_10](https://doi.org/10.1007/978-3-031-94578-6_10)


--------

# Reusable Reranking in OntoAligner

This notebook demonstrates how [reranking](https://ontoaligner.readthedocs.io/aligner/retriever.html#reranking) can be used as a reusable candidate-refinement step in OntoAligner. Reranking is not tied to one specific aligner. It can be added after any component that produces multiple target candidates for a source concept.

We start with the standard retrieval-based workflow, then show how the same idea can be applied to graph-based candidates, RAG retrieval outputs before LLM verification, and flat `source`–`target`–`score` predictions.

Together, these examples show how reranking can be reused across different OntoAligner workflows when candidate alignments need to be refined before producing the final matchings.

---
Contents of this tutorial:

1. Retrieval-based candidate reranking
2. Graph-based candidate reranking
3. Flat-output reranking
4. RAG IR-Output Reranking Before LLM Verification


---
## 1️⃣ Retrieval-Based Candidate Reranking

We start with the most direct reranking workflow in OntoAligner: reranking candidates produced by a [retrieval aligner](https://ontoaligner.readthedocs.io/aligner/retriever.html#retrieval).

A retrieval model first searches the target ontology and returns several possible target candidates for each source concept. In this example, `SBERTRetrieval` is used to generate those candidates.

The retrieval output is already grouped by source concept, which is the format expected by the reranker:

```python
{
    "source": source_iri,
    "target-cands": [target_iri_1, target_iri_2, ...],
    "score-cands": [score_1, score_2, ...],
}
```
Because the candidates are already grouped, they can be passed directly to CrossEncoderReranking or CohereReranking. The reranker scores the candidate pairs again and returns the same grouped format, with the target candidates reordered.

The flow below shows how retrieval candidates are generated, reranked, postprocessed, and converted into final matchings.


```text
Concept encoder
        ↓
SBERTRetrieval
        ↓
Grouped retrieval candidates
        ↓
CrossEncoderReranking / CohereReranking
        ↓
Reranked candidate groups
        ↓
retriever_postprocessor
        ↓
Final matchings

```

In [ ]:
# Import necessary libraries
import json
import os
import torch

# Import necessary modules from the 'ontoaligner' library
# The library provides tools for ontology alignment tasks, including dataset management,
# encoding, retrieval, reranking, evaluation, and postprocessing.
from ontoaligner.encoder import ConceptParentLightweightEncoder
from ontoaligner.ontology import MaterialInformationMatOntoOMDataset
from ontoaligner.utils import metrics, xmlify
from ontoaligner.aligner import SBERTRetrieval, CohereReranking, CrossEncoderReranking
from ontoaligner.postprocess import retriever_postprocessor


# Step 1: Initialize the ontology matching task
# The task is created using the Material Information Ontology Dataset,
# which includes source and target ontologies and reference matchings for evaluation.
task = MaterialInformationMatOntoOMDataset()

# Confirm the task initialization by printing its details
print("Test Task:", task)

# Step 2: Collect the ontology dataset
# The dataset includes paths to the source ontology, target ontology, and reference matching files.
dataset = task.collect(
    source_ontology_path="assets/MI-MatOnto/mi_ontology.xml",
    target_ontology_path="assets/MI-MatOnto/matonto_ontology.xml",
    reference_matching_path="assets/MI-MatOnto/matchings.xml",
)

# Step 3: Select the runtime device
# The reranking example can use GPU if CUDA is available; otherwise, it runs on CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Step 4: Initialize the encoder model
# The encoder prepares source and target ontology concept texts for retrieval and reranking.
encoder_model = ConceptParentLightweightEncoder()

# Encode the source and target ontologies
# The output is used as input for the retrieval model.
source_onto, target_onto = encoder_model(source=dataset["source"],target=dataset["target"])

# Step 5: Set up the retrieval model
# The retrieval model generates candidate target concepts for each source concept.
# Here, SBERTRetrieval is used with the all-MiniLM-L6-v2 sentence-transformer model.
retriever = SBERTRetrieval(device=device, top_k=10)

# Load the SBERT retrieval model
retriever.load(path="all-MiniLM-L6-v2")

# Generate candidate alignments
# The retrieval model returns grouped candidates using source, target-cands, and score-cands.
retrieval_outputs = retriever.generate(input_data=[source_onto,target_onto])

# Step 6: Set up the reranking model
# The reranker refines the candidates generated by the retrieval model.
# CrossEncoderReranking scores each source-target pair jointly and reranks the retrieved candidates.
reranker = CrossEncoderReranking(
    device=device,
    top_k=5,
    normalize_score="sigmoid",
)

# Load the CrossEncoder reranking model
reranker.load(path="cross-encoder/ms-marco-MiniLM-L6-v2")

# To use Cohere reranking instead of CrossEncoderReranking, replace the block above with:
# reranker = CohereReranking(
#     cohere_key=os.environ["COHERE_API_KEY"],
#     top_k=5,
#     normalize_score="none",
# )
# # Load the Cohere reranking model
# reranker.load(path="rerank-v3.5")

# Step 7: Rerank the retrieved candidates
# The reranker preserves the retrieval output format, so the existing retriever_postprocessor
# can still be used after reranking.
reranked_outputs = reranker.generate(input_data=[source_onto,target_onto,retrieval_outputs])

# Step 8: Post-process the reranked outputs
# The retriever_postprocessor converts grouped candidates into flat source-target matchings.
# For CrossEncoderReranking with sigmoid normalization, threshold=0.5 can be used.
# For CohereReranking, threshold=0.0 keeps the reranked top-k candidates.
matchings = retriever_postprocessor(predicts=reranked_outputs,threshold=0.5)

# Step 9: Evaluate the generated matchings
# The evaluation report compares predicted matchings against the reference alignments
# using metrics such as precision, recall, and F-score.
evaluation = metrics.evaluation_report(predicts=matchings,references=dataset["reference"])

# Print the evaluation report in a human-readable JSON format
print("Evaluation Report:", json.dumps(evaluation, indent=4))

# Step 10: Export matchings in XML format or JSON format

# XML format
# Convert the generated matchings into an XML alignment file using the xmlify utility.
xml_str = xmlify.xml_alignment_generator(matchings=matchings)

# Save the XML alignment to a file for further use or analysis
output_file_path = "reranked_matchings.xml"
with open(output_file_path, "w", encoding="utf-8") as xml_file:
    xml_file.write(xml_str)

print(f"Matchings in XML format have been successfully written to '{output_file_path}'.")

# JSON format
# Save the generated matchings in dictionary format for further analysis or debugging.
output_file_path = "reranked_matchings.json"
with open(output_file_path, "w", encoding="utf-8") as json_file:
    json.dump(matchings, json_file, indent=4, ensure_ascii=False)

print(f"Matchings in JSON format have been successfully written to '{output_file_path}'.")

---
## 2️⃣ Graph-Based Candidate Reranking

This section shows how reranking can be used after a [graph-based KGE aligner](https://ontoaligner.readthedocs.io/aligner/kge.html#knowledge-graph-embedding).

Unlike retrieval models, graph KGE aligners generate candidates from ontology triples and learned graph embeddings. In this example, `ConvEAligner` is used in retrieval mode by setting `retriever=True`. This makes the graph aligner return multiple target candidates for each source entity instead of only one final prediction.

The reranker is then applied to these graph-generated candidates. Since `CrossEncoderReranking` compares text pairs, the source and target ontologies are also encoded into text format. The graph aligner provides the candidate IRIs, and the text encoder provides the source and target descriptions used for reranking.

The workflow below shows how graph-based candidates are generated, reranked, and converted into final matchings.


```text
GraphTripleOMDataset
        ↓
GraphTripleEncoder
        ↓
ConvEAligner(retriever=True)
        ↓
Grouped graph-generated candidates
        ↓
CrossEncoderReranking
        ↓
Reranked graph candidates
        ↓
retriever_postprocessor
        ↓
Final matchings
```

In [ ]:
# Import necessary libraries
import json
import torch

# Import necessary modules from the ontoaligner package
from ontoaligner.ontology import GraphTripleOMDataset, MaterialInformationMatOntoOMDataset
from ontoaligner.encoder import GraphTripleEncoder, ConceptParentLightweightEncoder
from ontoaligner.aligner import ConvEAligner, CrossEncoderReranking
from ontoaligner.postprocess import retriever_postprocessor
from ontoaligner.utils import metrics, xmlify


def print_matchings(title, matchings, limit=10):
    """
    Prints a small sample of matchings in readable JSON format.
    """
    print(f"\n{title}")
    print(f"Total matchings: {len(matchings)}")
    print(json.dumps(matchings[:limit], indent=4, ensure_ascii=False))


# Step 1: Initialize graph ontology matching task
# GraphTripleOMDataset parses the ontology into triples for the KGE aligner.
graph_task = GraphTripleOMDataset(ontology_name="MI-MatOnto")
print("Graph task:", graph_task)

# Step 2: Load source, target, and reference ontologies in graph triple format
graph_dataset = graph_task.collect(
    source_ontology_path="assets/MI-MatOnto/mi_ontology.xml",
    target_ontology_path="assets/MI-MatOnto/matonto_ontology.xml",
    reference_matching_path="assets/MI-MatOnto/matchings.xml",
)

print("Graph dataset key-values:", graph_dataset.keys())
print("Sample graph source item:", graph_dataset["source"][0])

# Step 3: Encode the dataset into graph triple format
graph_encoder = GraphTripleEncoder()
encoded_graph_dataset = graph_encoder(**graph_dataset)

# Step 4: Load the same ontology files in standard ontology format
# This is only used to create source and target text for the text-based reranker.
text_task = MaterialInformationMatOntoOMDataset()

text_dataset = text_task.collect(
    source_ontology_path="assets/MI-MatOnto/mi_ontology.xml",
    target_ontology_path="assets/MI-MatOnto/matonto_ontology.xml",
    reference_matching_path="assets/MI-MatOnto/matchings.xml",
)

# Step 5: Encode source and target concepts into text format for reranking
# The graph aligner works from triples, while CrossEncoderReranking compares text pairs.
text_encoder = ConceptParentLightweightEncoder()

source_onto, target_onto = text_encoder(
    source=text_dataset["source"],
    target=text_dataset["target"],
)

# Step 6: Select runtime device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Step 7: Define training parameters for the graph KGE aligner
kge_params = {
    "device": "cpu",
    "embedding_dim": 300,

    # A small number of epochs keeps the example lightweight.
    # Increase this value for stronger graph embeddings.
    "num_epochs": 3,

    "train_batch_size": 128,
    "eval_batch_size": 64,
    "num_negs_per_pos": 5,
    "random_seed": 42,

    # Important for reranking:
    # retriever=True returns multiple target candidates per source.
    "retriever": True,
    "top_k": 10,
}

# Step 8: Initialize and run the graph KGE aligner
aligner = ConvEAligner(model="ConvE", **kge_params)

# Because retriever=True, the graph aligner returns grouped candidate outputs:
# {
#     "source": source_iri,
#     "target-cands": [...],
#     "score-cands": [...],
# }
graph_candidates = aligner.generate(input_data=encoded_graph_dataset)

# Step 9: Post-process and print graph candidates before reranking
graph_matchings = retriever_postprocessor(
    predicts=graph_candidates,
    threshold=0.5,
)

print_matchings(
    title="Graph KGE Matchings -- before reranking",
    matchings=graph_matchings,
    limit=10,
)

# Step 10: Initialize the reranking model
reranker = CrossEncoderReranking(
    device=device,
    top_k=5,
    normalize_score="sigmoid",
)

reranker.load(path="cross-encoder/ms-marco-MiniLM-L6-v2")

# Step 11: Rerank graph-generated candidates
# The graph aligner provides candidate IRIs.
# source_onto and target_onto provide text for those IRIs.
reranked_graph_candidates = reranker.generate(
    input_data=[
        source_onto,
        target_onto,
        graph_candidates,
    ]
)

# Step 12: Post-process and print reranked graph candidates
reranked_graph_matchings = retriever_postprocessor(
    predicts=reranked_graph_candidates,
    threshold=0.3,
)

print_matchings(
    title="Graph KGE Matchings -- after reranking",
    matchings=reranked_graph_matchings,
    limit=10,
)

print("Number of graph matchings before reranking:", len(graph_matchings))
print("Number of graph matchings after reranking:", len(reranked_graph_matchings))

# Step 13: Evaluate reranked graph candidates
evaluation = metrics.evaluation_report(
    predicts=reranked_graph_matchings,
    references=graph_dataset["reference"],
)

print("Graph KGE Reranking Evaluation Report:")
print(json.dumps(evaluation, indent=4))

# Step 14: Save reranked matchings in XML format
xml_str = xmlify.xml_alignment_generator(matchings=reranked_graph_matchings)

with open("graph_reranked_matchings.xml", "w", encoding="utf-8") as xml_file:
    xml_file.write(xml_str)

print("Reranked graph matchings have been written to 'graph_reranked_matchings.xml'.")

---

## 3️⃣ Flat-Output Reranking

Flat-output reranking applies to aligners that return flat `source`–`target`–`score` predictions.

In this section, [OLaLA](https://ontoaligner.readthedocs.io/aligner/olala.html) is used to show the flat-output case. The same idea also applies to ensemble outputs, FLORA-style outputs, custom aligners, and final LLM/RAG-style outputs.

Some aligners, such as OLaLA, return flat predictions directly. RAG-style workflows usually need their postprocessor first, such as `rag_hybrid_postprocessor` or `rag_heuristic_postprocessor`, to convert IR and LLM outputs into final flat matchings.

Before reranking, flat predictions are grouped by source concept. After grouping, they follow the same candidate format used by retrieval-based reranking and can be passed to `CrossEncoderReranking`.

This strategy is most useful when the flat output contains multiple target candidates for the same source concept. If an aligner already returns only one final target per source, there may be nothing meaningful left to rerank.

The workflow below shows how flat alignment outputs are converted into grouped candidates and reranked.

```text
Aligner with flat output
        ↓
Flat source-target-score alignments
        ↓
[For RAG-style workflows: RAG/model-specific postprocessor first]
        ↓
group_predictions_for_reranking
        ↓
Grouped candidate format
        ↓
CrossEncoderReranking
        ↓
retriever_postprocessor
        ↓
Final reranked matchings
```

In [ ]:
# Run OLaLa flat-output reranking in OntoAligner

import json
import torch

from ontoaligner.ontology import OLaLaOMDataset
from ontoaligner.encoder import OLaLaEncoder
from ontoaligner.aligner import CrossEncoderReranking
from ontoaligner.aligner.olala import (
    OLaLaSBERTRetrieval,
    OLaLaLLMAligner,
    OLaLaHighPrecisionMatcher,
    OLaLaAligner,
)
from ontoaligner.aligner.olala.postprocessor import olala_postprocessor
from ontoaligner.postprocess import retriever_postprocessor
from ontoaligner.utils import metrics, xmlify


def print_matchings(title, matchings, limit=10):
    """
    Prints a small sample of matchings in readable JSON format.
    """
    print(f"\n{title}")
    print(f"Total matchings: {len(matchings)}")
    print(json.dumps(matchings[:limit], indent=4, ensure_ascii=False))


def group_predictions_for_reranking(predictions):
    """
    Converts flat source-target-score predictions into grouped candidate format.

    Input format:
    [
        {"source": source_iri, "target": target_iri, "score": score},
        ...
    ]

    Output format:
    [
        {
            "source": source_iri,
            "target-cands": [target_iri_1, target_iri_2, ...],
            "score-cands": [score_1, score_2, ...],
        },
        ...
    ]
    """
    grouped_predictions = {}

    for prediction in predictions:
        source = prediction["source"]
        target = prediction["target"]
        score = prediction.get("score", 0.0)

        if source not in grouped_predictions:
            grouped_predictions[source] = {
                "source": source,
                "target-cands": [],
                "score-cands": [],
            }

        grouped_predictions[source]["target-cands"].append(target)
        grouped_predictions[source]["score-cands"].append(float(score))

    return list(grouped_predictions.values())


# Step 1: Load task and ontologies
task = OLaLaOMDataset()
print("Test Task:", task)

dataset = task.collect(
    source_ontology_path="assets/MI-MatOnto/mi_ontology.xml",
    target_ontology_path="assets/MI-MatOnto/matonto_ontology.xml",
    reference_matching_path="assets/MI-MatOnto/matchings.xml",
)

# Step 2: Encode ontologies
encoder_model = OLaLaEncoder()

encoded_ontology = encoder_model(
    source=dataset["source"],
    target=dataset["target"],
)

source_onto = encoded_ontology[0]
target_onto = encoded_ontology[1]

# Step 3: SBERT candidate generation
retriever = OLaLaSBERTRetrieval(
    device="cuda",
    top_k=5,
    both_directions=True,
    topk_per_resource=True,
)

# Step 4: LLM binary verification
llm_aligner = OLaLaLLMAligner(
    device="cuda",
    max_new_tokens=10,
    temperature=0.0,
    truncation=True,
    max_length=2048,
    padding=True,
    loading_arguments={
        "device_map": "auto",
        "torch_dtype": torch.float16,
    },
)

# Step 5: High-precision matcher
hp_aligner = OLaLaHighPrecisionMatcher(confidence=1.0)

# Step 6: Initialize OLaLa aligner
olala = OLaLaAligner(
    retriever=retriever,
    llm_aligner=llm_aligner,
    hp_aligner=hp_aligner,
)

olala.load(
    llm_path="upstage/Llama-2-70b-instruct-v2",
    retriever_path="multi-qa-mpnet-base-dot-v1",
)

# Step 7: Generate flat OLaLa alignments
# OLaLa returns flat source-target-score predictions with alignment_type.
alignments = olala.generate(input_data=encoded_ontology)

print_matchings(
    title="Flat OLaLa Alignments -- before reranking",
    matchings=alignments,
    limit=10,
)

# Optional: original OLaLa postprocessing before reranking
original_final_matchings = olala_postprocessor(
    alignments,
    encoded_ontology,
    confidence_threshold=0.5,
    strict_bad_hosts=False,
)

print_matchings(
    title="OLaLa Final Matchings -- before reranking",
    matchings=original_final_matchings,
    limit=10,
)

# Step 8: Convert flat OLaLa alignments into grouped candidate format
grouped_candidates = group_predictions_for_reranking(
    predictions=alignments,
)

print_matchings(
    title="Grouped OLaLa Candidates -- ready for reranking",
    matchings=grouped_candidates,
    limit=10,
)

# Step 9: Initialize reranking model
device = "cuda" if torch.cuda.is_available() else "cpu"

reranker = CrossEncoderReranking(
    device=device,
    top_k=5,
    normalize_score="sigmoid",
)

reranker.load(path="cross-encoder/ms-marco-MiniLM-L6-v2")

# Step 10: Rerank grouped OLaLa candidates
reranked_outputs = reranker.generate(
    input_data=[
        source_onto,
        target_onto,
        grouped_candidates,
    ]
)

print_matchings(
    title="Grouped OLaLa Candidates -- after reranking",
    matchings=reranked_outputs,
    limit=10,
)

# Step 11: Convert reranked grouped candidates into flat matchings
reranked_matchings = retriever_postprocessor(
    predicts=reranked_outputs,
    threshold=0.5,
)

print_matchings(
    title="Flat OLaLa Matchings -- after reranking",
    matchings=reranked_matchings,
    limit=10,
)

print("Number of flat OLaLa alignments before reranking:", len(alignments))
print("Number of original OLaLa final matchings:", len(original_final_matchings))
print("Number of grouped sources for reranking:", len(grouped_candidates))
print("Number of reranked OLaLa matchings:", len(reranked_matchings))

# Step 12: Evaluate original OLaLa final matchings
original_evaluation = metrics.evaluation_report(
    predicts=original_final_matchings,
    references=dataset["reference"],
)

print("Original OLaLa Evaluation Report:")
print(json.dumps(original_evaluation, indent=4))

# Step 13: Evaluate reranked OLaLa matchings
reranked_evaluation = metrics.evaluation_report(
    predicts=reranked_matchings,
    references=dataset["reference"],
)

print("OLaLa Flat-output Reranking Evaluation Report:")
print(json.dumps(reranked_evaluation, indent=4))

# Step 14: XML export
xml_str = xmlify.xml_alignment_generator(matchings=reranked_matchings)

output_file_path = "olala_reranked_matchings.xml"
with open(output_file_path, "w", encoding="utf-8") as xml_file:
    xml_file.write(xml_str)

print(f"Saved reranked OLaLa matchings to {output_file_path}")

---
## 4️⃣ RAG IR-Output Reranking Before LLM Verification

This section shows how reranking can be inserted inside a [RAG aligner](https://ontoaligner.readthedocs.io/aligner/rag.html).

In a RAG workflow, the retriever first generates candidate source-target pairs. These candidates are then passed to an LLM for verification. Reranking can be added between these two steps, so the LLM receives a smaller and better-ranked set of candidates.

The same pattern can be followed by other RAG-style aligners such as FewShotRAG, ICV, or any custom RAG pipeline where retrieval and LLM verification are separate steps.

The workflow below shows how IR candidates are reranked before LLM verification.

```text
ConceptParentRAGEncoder
        ↓
RAG retriever
        ↓
IR candidate groups
        ↓
CrossEncoderReranking
        ↓
Reranked IR candidates
        ↓
LLM verification
        ↓
RAG postprocessing
        ↓
Final matchings
```

In [ ]:
# Import required libraries and modules
import json
import torch

from ontoaligner.ontology import MaterialInformationMatOntoOMDataset
from ontoaligner.encoder import ConceptParentRAGEncoder
from ontoaligner.postprocess import (
    retriever_postprocessor,
    rag_hybrid_postprocessor,
    rag_heuristic_postprocessor,
)
from ontoaligner.utils import metrics, xmlify

# Direct imports are used to build the RAG pipeline explicitly.
from ontoaligner.aligner.rag.rag import RAG, AutoModelDecoderRAGLLMV2
from ontoaligner.aligner.retrieval.models import SBERTRetrieval
from ontoaligner.aligner.retrieval.reranking import CrossEncoderReranking


def print_matchings(title, matchings, limit=10):
    """
    Prints a small sample of matchings in readable JSON format.
    """
    print(f"\n{title}")
    print(f"Total matchings: {len(matchings)}")
    print(json.dumps(matchings[:limit], indent=4, ensure_ascii=False))


# Step 1: Initialize the dataset object for MaterialInformation MatOnto dataset
task = MaterialInformationMatOntoOMDataset()
print("Test Task:", task)

# Step 2: Load source and target ontologies along with reference matchings
dataset = task.collect(
    source_ontology_path="assets/MI-MatOnto/mi_ontology.xml",
    target_ontology_path="assets/MI-MatOnto/matonto_ontology.xml",
    reference_matching_path="assets/MI-MatOnto/matchings.xml",
)

# Step 3: Encode the source and target ontologies
# ConceptParentRAGEncoder prepares the input structure used by the RAG pipeline.
encoder_model = ConceptParentRAGEncoder()
encoded_ontology = encoder_model(
    source=dataset["source"],
    target=dataset["target"],
)

# Step 4: Select runtime device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Step 5: Define configuration for the retriever and LLM
retriever_config = {
    "device": device,
    "top_k": 10,
    "threshold": 0.1,
}

llm_config = {
    # CPU keeps the example easier to run.
    # Change to "cuda" if your environment supports the selected LLM.
    "device": "cpu",
    "max_length": 300,
    "max_new_tokens": 10,
    "huggingface_access_token": "",
    "device_map": "balanced",
    "batch_size": 15,
    "answer_set": {
        "yes": ["yes", "correct", "true", "positive", "valid"],
        "no": ["no", "incorrect", "false", "negative", "invalid"],
    },
}

# Step 6: Initialize the RAG-based ontology matcher
model = RAG(
    retriever=SBERTRetrieval,
    llm=AutoModelDecoderRAGLLMV2,
    retriever_config=retriever_config,
    llm_config=llm_config,
)

# A small decoder model is used for a lightweight runnable example.
# For the original Mistral setup, replace "distilgpt2" with:
# "mistralai/Mistral-7B-v0.3"
model.load(
    llm_path="distilgpt2",
    ir_path="all-MiniLM-L6-v2",
)

# Decoder-only models generate more correctly with left padding.
# This avoids the right-padding warning for lightweight models such as distilgpt2.
model.LLM.tokenizer.padding_side = "left"

if model.LLM.tokenizer.pad_token is None:
    model.LLM.tokenizer.pad_token = model.LLM.tokenizer.eos_token

model.LLM.model.config.pad_token_id = model.LLM.tokenizer.pad_token_id

# Step 7: Build the retriever input used inside the RAG model
# This gives source and target concept text needed by the reranker.
retrieval_input = encoded_ontology["retriever-encoder"]()(
    **encoded_ontology["task-args"]
)

source_onto = retrieval_input[0]
target_onto = retrieval_input[1]

# Step 8: Generate IR candidates using the RAG retriever
# The IR output is grouped candidate format:
# {
#     "source": source_iri,
#     "target-cands": [...],
#     "score-cands": [...],
# }
ir_outputs = model.Retrieval.generate(
    input_data=retrieval_input
)

# Step 9: Print general IR alignments before reranking
general_ir_matchings = retriever_postprocessor(
    predicts=ir_outputs,
    threshold=retriever_config["threshold"],
)

print_matchings(
    title="General IR Matchings -- before reranking",
    matchings=general_ir_matchings,
    limit=10,
)

# Step 10: Initialize the reranking model
reranker = CrossEncoderReranking(
    device=device,
    top_k=5,
    normalize_score="sigmoid",
)

reranker.load(path="cross-encoder/ms-marco-MiniLM-L6-v2")

# Step 11: Rerank the IR candidates before LLM verification
reranked_ir_outputs = reranker.generate(
    input_data=[
        source_onto,
        target_onto,
        ir_outputs,
    ]
)

# Step 12: Convert reranked grouped candidates into flat source-target pairs
# The LLM verification step expects retrieved source-target pairs.
reranked_ir_matchings = retriever_postprocessor(
    predicts=reranked_ir_outputs,
    threshold=0.5,
)

print_matchings(
    title="Reranked IR Matchings -- sent to LLM",
    matchings=reranked_ir_matchings,
    limit=10,
)

print("Number of original IR candidate groups:", len(ir_outputs))
print("Number of general IR matchings:", len(general_ir_matchings))
print("Number of reranked IR matchings sent to LLM:", len(reranked_ir_matchings))

# Step 13: Send reranked IR matchings to the LLM verification step
llm_predictions = model.llm_generate(
    input_data=encoded_ontology,
    ir_output=reranked_ir_matchings,
)

print_matchings(
    title="Final LLM Predictions -- after reranked IR",
    matchings=llm_predictions,
    limit=10,
)

# Step 14: Build RAG-style output with reranked IR candidates
# This keeps the output compatible with existing RAG postprocessors.
predicts = [
    {"ir-outputs": reranked_ir_outputs},
    {"llm-output": llm_predictions},
]

# Step 15: Apply heuristic postprocessing
heuristic_matchings, heuristic_configs = rag_heuristic_postprocessor(
    predicts=predicts,
    topk_confidence_ratio=3,
    topk_confidence_score=3,
)

print_matchings(
    title="Heuristic Final Matchings",
    matchings=heuristic_matchings,
    limit=10,
)

evaluation = metrics.evaluation_report(
    predicts=heuristic_matchings,
    references=dataset["reference"],
)

print("RAG with Reranked IR -- Heuristic Matching Evaluation Report:")
print(json.dumps(evaluation, indent=4))
print("Heuristic Matching Obtained Configuration:", heuristic_configs)

# Step 16: Apply hybrid postprocessing
hybrid_matchings, hybrid_configs = rag_hybrid_postprocessor(
    predicts=predicts,
    ir_score_threshold=0.5,
    llm_confidence_th=0.8,
)

print_matchings(
    title="Hybrid Final Matchings",
    matchings=hybrid_matchings,
    limit=10,
)

evaluation = metrics.evaluation_report(
    predicts=hybrid_matchings,
    references=dataset["reference"],
)

print("RAG with Reranked IR -- Hybrid Matching Evaluation Report:")
print(json.dumps(evaluation, indent=4))
print("Hybrid Matching Obtained Configuration:", hybrid_configs)

# Step 17: Convert hybrid matchings to XML format
xml_str = xmlify.xml_alignment_generator(matchings=hybrid_matchings)

output_file_path = "rag_reranked_matchings.xml"
with open(output_file_path, "w", encoding="utf-8") as xml_file:
    xml_file.write(xml_str)

print(f"RAG reranked matchings have been written to '{output_file_path}'.")

## ✅ Key Takeaways

- Reranking is a reusable candidate-refinement step in OntoAligner.
- It can be applied after aligners that produce multiple target candidates for each source concept.
- Grouped outputs with `target-cands` and `score-cands` can be reranked directly.
- Flat outputs with `source`, `target`, and `score` can be grouped by source concept before reranking.
- The same reranking pattern can be used across retrieval, graph-based, RAG-style, ensemble, OLaLA, and custom aligner workflows.

For more information, visit the [OntoAligner Documentation](https://ontoaligner.readthedocs.io/)

-----------------------------------------------------------
-----------------------------------------------------------

📃 Acknowledgement

OntoAligner is licensed under [![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)


```bibtex
@inproceedings{babaei2025ontoaligner,
  title={OntoAligner: A Comprehensive Modular and Robust Python Toolkit for Ontology Alignment},
  author={Babaei Giglou, Hamed and D’Souza, Jennifer and Karras, Oliver and Auer, S{\"o}ren},
  booktitle={European Semantic Web Conference},
  pages={174--191},
  year={2025},
  organization={Springer}
}
```